# 01 · SNOOPPI: frozen ProtT5 interaction predictor

This notebook learns a binary classifier for the **SNOOPPI positive versus negative pair labels**, using frozen ProtT5 protein representations and a symmetric multilayer perceptron (MLP). It also creates the train/validation/test CSV files consumed by notebooks 02 and 03.

**Preservation policy.** All 42 original code cells, their source text, and their order are preserved. Numbered explanations have been added; saved outputs, execution counts, and transient Colab/widget metadata have been removed. The notebook has been checked structurally but **has not been rerun** during repository preparation. Existing exploratory, repeated-training, and Drive-recovery code remains explicitly labeled.

| Item | Definition in the preserved implementation |
| --- | --- |
| Supervised input | `positive` and `negative` splits from `ChatterjeeLab/SNOOPPI`; unknown pairs are excluded |
| Target | `1` when `SNOOPPI_final_label == "positive"`; `0` otherwise within those two input splits |
| Split | Stratified pair-row random split, approximately 80% / 10% / 10%, `random_state=44` |
| Encoder | `Rostlab/prot_t5_xl_half_uniref50-enc`, frozen, loaded in float16 |
| Representation | First 1,024 characters after uppercasing and replacing `U/Z/O/B` with `X`; mean pooling excludes padding and final EOS |
| Pair features | Concatenation of sum, absolute difference, and elementwise product of the two 1,024-dimensional vectors |
| Classifier | `3072 → 512 → 512 → 256 → 1`; ReLU and dropout 0.1 after hidden layers |
| Optimization | AdamW, learning rate `1e-4`, weight decay `1e-4`, batch size 1,024; weighted binary cross-entropy |
| Selection | Up to 50 epochs; early-stopping patience 10; best validation macro average precision (macro-AP) |
| Decision threshold | Validation macro-F1 maximization over 181 thresholds from 0.05 to 0.95 |

**Execution guide.** This is a Colab-oriented research notebook with downloads, GPU inference, and Google Drive writes. For one seeded training run, use sections **03–08**, skip the first historical loop in **09**, then use **10–12**. Sections **01–02** preserve earlier data exploration; **13–17** preserve optional diagnostics, unknown-data archiving, recovery, and table export. Run these optional cells only when their documented conditions apply. Review each output path before execution; existing files can be overwritten.

**Interpretation.** The main training path reloads the original labeled splits and **does not consume the earlier `cleaned` dataset**. It does not establish pair, protein, or sequence-cluster disjointness. Results therefore concern this pair-row split and do not demonstrate performance on unseen proteins. The tokenizer normalization is not a biological validation or post-translational-modification parser. Dataset/model revisions and package versions are not pinned in this preserved notebook; record them for any reported experiment.

## 01. Data access and streaming inspection

Install data-access packages and inspect one example from each Hugging Face split. This is an exploratory view of the source schema and label categories, not the training dataset. `streaming=True` avoids materializing the complete dataset in this step. Package installation uses upgrades and is not version pinned.

In [ ]:
!pip -q install -U datasets huggingface_hub pyarrow pandas

In [ ]:
from datasets import load_dataset

snooppi_stream = load_dataset(
    "ChatterjeeLab/SNOOPPI",
    streaming=True,
)

print(snooppi_stream)

In [ ]:
for split_name in ["positive", "negative", "unknown"]:
    example = next(iter(snooppi_stream[split_name]))

    print(f"\n--- {split_name.upper()} ---")
    print("Label:", example["SNOOPPI_final_label"])
    print("Partner A length:", example["partner_A_length"])
    print("Partner B length:", example["partner_B_length"])
    print("Partner A type:", example["partner_A_mol_type"])
    print("Partner B type:", example["partner_B_mol_type"])
    print("Interaction type:", example["unique_interaction_mi_terms"])

## 02. Historical filtered-data export

Load positive and negative pairs, retain selected metadata columns, shuffle with seed 42, and produce an exploratory filtered dataset. The predicate requires both molecule types to be `protein`, allowed sequence characters, and each partner length between 30 and 1,500. The allowed alphabet includes ambiguous/nonstandard residue letters as coded.

The positive split is loaded twice, including a separate `/content/hf_cache` attempt; both original cells are retained. `cleaned.save_to_disk()` writes to `/content/drive/MyDrive/SNOOPPI_project/clean_binary_dataset`, but this section does not mount Google Drive. Verify the mount before treating that path as persistent. The final preview cell returns to streaming examples.

**This export is not used by the supervised training path below.** Its filtering and seed 42 must not be described as the preprocessing or split seed of the reported classifier.

In [ ]:
positive = load_dataset(
    "ChatterjeeLab/SNOOPPI",
    split="positive",
)

negative = load_dataset(
    "ChatterjeeLab/SNOOPPI",
    split="negative",
)

print(positive)
print(negative)

In [ ]:
positive = load_dataset(
    "ChatterjeeLab/SNOOPPI",
    split="positive",
    cache_dir="/content/hf_cache",
)

In [ ]:
KEEP_COLUMNS = [
    "SNOOPPI_Pair_ID",
    "partner_A_sequence",
    "partner_B_sequence",
    "partner_A_length",
    "partner_B_length",
    "partner_A_mol_type",
    "partner_B_mol_type",
    "partner_A_species",
    "partner_B_species",
    "unique_interaction_mi_terms",
    "SNOOPPI_final_label",
]

positive_small = positive.select_columns(KEEP_COLUMNS)
negative_small = negative.select_columns(KEEP_COLUMNS)

In [ ]:
from datasets import concatenate_datasets

dataset = concatenate_datasets([
    positive_small,
    negative_small,
])

dataset = dataset.shuffle(seed=42)

print(dataset)

In [ ]:
import re

AA_PATTERN = re.compile(r"^[ACDEFGHIKLMNPQRSTVWYBXZJUO]+$")

def valid_pair(row):
    seq_a = row["partner_A_sequence"]
    seq_b = row["partner_B_sequence"]

    return (
        row["partner_A_mol_type"] == "protein"
        and row["partner_B_mol_type"] == "protein"
        and isinstance(seq_a, str)
        and isinstance(seq_b, str)
        and AA_PATTERN.fullmatch(seq_a) is not None
        and AA_PATTERN.fullmatch(seq_b) is not None
        and 30 <= len(seq_a) <= 1500
        and 30 <= len(seq_b) <= 1500
    )

cleaned = dataset.filter(valid_pair, num_proc=2)
print(dataset.num_rows, cleaned.num_rows)

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/SNOOPPI_project/clean_binary_dataset"

cleaned.save_to_disk(OUTPUT_DIR)

In [ ]:
import pandas as pd

preview = list(snooppi_stream["positive"].take(5))
df_preview = pd.DataFrame(preview)

df_preview.head()

## 03. Load the source splits for the main experiment

Materialize the complete Hugging Face dataset and print each split's size. The training dataframe in section 05 is constructed from `snooppi["positive"]` and `snooppi["negative"]`.

The size-inspection loop also leaves `split_data` bound to its last iterated split. The optional archive in section 14 later relies on that notebook variable; its identity must be checked before reuse. The `unknown` split is loaded here but is not added to the supervised dataframe.

In [ ]:
from datasets import load_dataset

snooppi = load_dataset("ChatterjeeLab/SNOOPPI")

In [ ]:
for split_name, split_data in snooppi.items():
    print(f"{split_name}: {split_data.num_rows:,} rows")

## 04. Runtime, dependencies, and frozen encoder

Install transformer dependencies, import training/evaluation packages, select CUDA when available, and load the ProtT5 tokenizer and encoder. `encoder.eval()` disables training behavior and `requires_grad=False` freezes all encoder parameters.

The model is explicitly loaded in float16 even if the selected device is CPU. That CPU fallback and environment compatibility have not been validated here. Use the original supported Colab/GPU environment when reproducing the run, and record resolved package versions and model revision.

In [ ]:
!pip -q install transformers sentencepiece accelerate scikit-learn

In [ ]:
import re
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5EncoderModel

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

In [ ]:
MODEL_ID = "Rostlab/prot_t5_xl_half_uniref50-enc"

tokenizer = T5Tokenizer.from_pretrained(
    MODEL_ID,
    do_lower_case=False
)

encoder = T5EncoderModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to(DEVICE)

encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

## 05. Binary labels and stratified pair-row partitions

Build a dataframe containing the two protein sequences and the SNOOPPI label from the original positive and negative splits. `label=1` denotes a positive pair, and `label=0` denotes a negative pair. Unknown pairs are excluded.

Two stratified `train_test_split` calls create approximately 80% training, 10% validation, and 10% test sets with seed 44. The subsequent cells inspect class counts and plot class composition. This code performs no canonical-pair deduplication, protein-disjoint assignment, or sequence-cluster isolation; those properties are not established by stratification. The early exploratory sequence/type filters in section 02 are not applied here.

In [ ]:
import pandas as pd
import numpy as np

KEEP_COLUMNS = [
    "partner_A_sequence",
    "partner_B_sequence",
    "SNOOPPI_final_label"
]

positive_df = (
    snooppi["positive"]
    .select_columns(KEEP_COLUMNS)
    .to_pandas()
)

negative_df = (
    snooppi["negative"]
    .select_columns(KEEP_COLUMNS)
    .to_pandas()
)

labeled_df = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)

labeled_df["label"] = (
    labeled_df["SNOOPPI_final_label"] == "positive"
).astype(np.float32)

print(labeled_df.shape)
print(labeled_df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, remaining_df = train_test_split(
    labeled_df,
    test_size=0.20,
    stratify=labeled_df["label"],
    random_state=44
)

val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    stratify=remaining_df["label"],
    random_state=44
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(df["label"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

split_counts = pd.DataFrame({
    "Train": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).T

split_counts = split_counts.rename(
    columns={
        0.0: "Negative",
        1.0: "Positive"
    }
)

display(split_counts)

ax = split_counts.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    color=["#E76F51", "#2A9D8F"]
)

ax.set_title("SNOOPPI Train, Validation, and Test Splits")
ax.set_xlabel("Split")
ax.set_ylabel("Number of interaction pairs")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Interaction label")

plt.tight_layout()
plt.show()

## 06. Normalize sequences and cache frozen embeddings

Normalize each original sequence string by uppercasing, replacing `U/Z/O/B` with `X`, and retaining the first 1,024 characters. Insert spaces for ProtT5 tokenization, encode batches of two, and mean-pool residue representations while excluding padding and the final EOS token. Cached values are CPU float16 tensors keyed by the original, unnormalized sequence strings.

The unique-sequence list includes the training, validation, and test partitions. Computing representations with the same frozen pretrained encoder does not use their interaction labels or fit the encoder to the test set. The embedding function creates a fresh in-memory cache and only saves after the complete call; it has no incremental resume/checkpoint mechanism. The local cache is written to `/content/prott5_snooppi_embeddings.pt` and is later copied into the artifact export.

The normalization code does not validate empty strings, modification markup, or ambiguous tokens and does not check for equivalent representations after truncation. These limitations are preserved and should be considered when interpreting the experiment.

In [ ]:
MAX_RESIDUES = 1024

def normalize_sequence(sequence):
    sequence = str(sequence).upper()
    sequence = re.sub(r"[UZOB]", "X", sequence)
    return sequence[:MAX_RESIDUES]

In [ ]:
@torch.inference_mode()
def embed_sequences(sequences, batch_size=2):
    embedding_cache = {}

    for start in range(0, len(sequences), batch_size):
        original_batch = sequences[start:start + batch_size]
        cleaned_batch = [
            normalize_sequence(sequence)
            for sequence in original_batch
        ]

        spaced_batch = [
            " ".join(list(sequence))
            for sequence in cleaned_batch
        ]

        tokens = tokenizer(
            spaced_batch,
            add_special_tokens=True,
            padding=True,
            return_tensors="pt"
        )

        input_ids = tokens["input_ids"].to(DEVICE)
        attention_mask = tokens["attention_mask"].to(DEVICE)

        output = encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden = output.last_hidden_state

        # Remove padding and the final EOS token
        residue_mask = attention_mask.bool().clone()
        final_token_positions = attention_mask.sum(dim=1) - 1

        residue_mask[
            torch.arange(len(original_batch), device=DEVICE),
            final_token_positions
        ] = False

        residue_mask_float = residue_mask.unsqueeze(-1).to(hidden.dtype)

        pooled = (
            hidden * residue_mask_float
        ).sum(dim=1) / residue_mask_float.sum(dim=1).clamp(min=1)

        pooled = pooled.detach().cpu().half()

        for sequence, embedding in zip(original_batch, pooled):
            embedding_cache[sequence] = embedding

        if start % 100 == 0:
            print(f"Embedded {start + len(original_batch):,}/{len(sequences):,}")

    return embedding_cache

In [ ]:
all_sequences = pd.unique(
    pd.concat([
        train_df["partner_A_sequence"],
        train_df["partner_B_sequence"],
        val_df["partner_A_sequence"],
        val_df["partner_B_sequence"],
        test_df["partner_A_sequence"],
        test_df["partner_B_sequence"]
    ], ignore_index=True)
).tolist()

print(f"Unique proteins: {len(all_sequences):,}")

In [ ]:
embedding_cache = embed_sequences(
    all_sequences,
    batch_size=2
)

In [ ]:
torch.save(
    embedding_cache,
    "/content/prott5_snooppi_embeddings.pt"
)

## 07. Symmetric pair features and data loaders

Retrieve the two frozen embeddings, convert them to float32, and construct

\[
x(A,B) = [h_A+h_B,\; |h_A-h_B|,\; h_A\odot h_B].
\]

These operations are unchanged by swapping the partner order, so the classifier receives 3,072 symmetric features per pair. Labels are float32 for binary cross-entropy. Training batches are shuffled; validation and test batches retain dataframe order. All loaders use batch size 1,024.

In [ ]:
class PPIPairDataset(Dataset):
    def __init__(self, dataframe, embedding_cache):
        self.df = dataframe.reset_index(drop=True)
        self.embedding_cache = embedding_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        h_a = self.embedding_cache[
            row["partner_A_sequence"]
        ].float()

        h_b = self.embedding_cache[
            row["partner_B_sequence"]
        ].float()

        pair_features = torch.cat([
            h_a + h_b,
            torch.abs(h_a - h_b),
            h_a * h_b
        ])

        label = torch.tensor(
            row["label"],
            dtype=torch.float32
        )

        return pair_features, label

In [ ]:
train_dataset = PPIPairDataset(train_df, embedding_cache)
val_dataset = PPIPairDataset(val_df, embedding_cache)
test_dataset = PPIPairDataset(test_df, embedding_cache)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False
)

## 08. Classifier, weighted loss, and epoch evaluation

Define the MLP and initialize AdamW with learning rate and weight decay both `1e-4`. The loss is `BCEWithLogitsLoss`, with positive weight equal to the training-set negative count divided by the positive count. This balances class contributions without relabeling or resampling examples.

`run_epoch()` trains or evaluates the head and returns loss, AUROC, positive-class AP, negative-class AP, and their arithmetic mean (`macro_ap`). Negative-class AP uses `1 - label` and `1 - score`. The later export reports **macro-AP**, which should not be confused with positive-class AP or a trapezoidal PR-curve area. Training-epoch metrics are collected while the model is updating and dropout is active; final split metrics are collected separately in evaluation mode.

In [ ]:
class PPIClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3072, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 1)
        )

    def forward(self, features):
        return self.network(features).squeeze(-1)

In [ ]:
model = PPIClassifier().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
n_positive = int(train_df["label"].sum())
n_negative = len(train_df) - n_positive

positive_weight = n_negative / n_positive

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [positive_weight],
        device=DEVICE
    )
)

print("Positive:", n_positive)
print("Negative:", n_negative)
print("Positive weight:", positive_weight)

In [ ]:
def run_epoch(model, loader, training):
    model.train(training)

    total_loss = 0
    all_labels = []
    all_probabilities = []

    for features, labels in loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad()

        logits = model(features)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            optimizer.step()

        probabilities = torch.sigmoid(logits)

        total_loss += loss.item() * len(labels)
        all_labels.extend(labels.detach().cpu().numpy())
        all_probabilities.extend(
            probabilities.detach().cpu().numpy()
        )

    labels = np.asarray(all_labels)
    probabilities = np.asarray(all_probabilities)

    positive_ap = average_precision_score(
        labels,
        probabilities
    )

    negative_ap = average_precision_score(
        1 - labels,
        1 - probabilities
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "auroc": roc_auc_score(labels, probabilities),
        "positive_ap": positive_ap,
        "negative_ap": negative_ap,
        "macro_ap": (positive_ap + negative_ap) / 2
    }

## 09. Historical first training loop

This original training loop runs up to 50 epochs, chooses the best validation macro-AP state, and stops after 10 consecutive non-improving epochs. It does not construct the `history_df` used by the later export and precedes the explicit training seed reset.

**For the documented single seeded run, skip this cell and continue to section 10.** Running both loops performs two training runs; the later reset discards this model and initializes a new head. The cell is preserved to retain the development history.

In [ ]:
best_state = None
best_macro_ap = -np.inf
patience = 10
epochs_without_improvement = 0

for epoch in range(1, 51):
    train_metrics = run_epoch(
        model,
        train_loader,
        training=True
    )

    with torch.no_grad():
        val_metrics = run_epoch(
            model,
            val_loader,
            training=False
        )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | "
        f"val AUROC {val_metrics['auroc']:.4f} | "
        f"val macro-AP {val_metrics['macro_ap']:.4f}"
    )

    if val_metrics["macro_ap"] > best_macro_ap:
        best_macro_ap = val_metrics["macro_ap"]
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

## 10. Seeded training run with history

Set PyTorch and NumPy seeds to 44, instantiate a **new** classifier and optimizer, then train with the same 50-epoch maximum, patience 10, and validation macro-AP selection rule. This reset is a fresh run, not a checkpoint resume.

The second cell records epoch-level history, restores the best head parameters from memory, and constructs `history_df`. These are the model and history consumed by sections 11–12. No on-disk optimizer/training-state checkpoint is saved during this loop; interruption recovery is not implemented here. Seeding alone does not establish deterministic behavior across devices and library versions.

In [ ]:
import copy
import numpy as np
import pandas as pd
import torch

torch.manual_seed(44)
np.random.seed(44)

model = PPIClassifier().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
history = []

best_state = None
best_macro_ap = -np.inf
patience = 10
epochs_without_improvement = 0

for epoch in range(1, 51):
    train_metrics = run_epoch(
        model,
        train_loader,
        training=True
    )

    with torch.no_grad():
        val_metrics = run_epoch(
            model,
            val_loader,
            training=False
        )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "val_loss": val_metrics["loss"],
        "val_auroc": val_metrics["auroc"],
        "val_macro_ap": val_metrics["macro_ap"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | "
        f"val AUROC {val_metrics['auroc']:.4f} | "
        f"val macro-AP {val_metrics['macro_ap']:.4f}"
    )

    if val_metrics["macro_ap"] > best_macro_ap:
        best_macro_ap = val_metrics["macro_ap"]
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

history_df = pd.DataFrame(history)

## 11. Inspect the logged training curves

Plot training/validation weighted loss and the validation AUROC and macro-AP from `history_df`. This cell displays figures only. Section 12 separately writes publication-oriented figure files to disk. The fixed score-axis range begins at 0.5, so values below that range would not be visible.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Training"
)

axes[0].plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation"
)

axes[0].set(
    title="Training and validation loss",
    xlabel="Epoch",
    ylabel="Weighted BCE loss"
)

axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(
    history_df["epoch"],
    history_df["val_auroc"],
    label="AUROC"
)

axes[1].plot(
    history_df["epoch"],
    history_df["val_macro_ap"],
    label="Macro-AP"
)

axes[1].set(
    title="Validation performance",
    xlabel="Epoch",
    ylabel="Score",
    ylim=(0.5, 1.0)
)

axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 12. Validation threshold, held-out evaluation, and artifact export

Collect predictions with the selected head in evaluation mode. Choose the decision threshold that maximizes **validation macro-F1** on the specified 0.05–0.95 grid; apply that threshold to the training, validation, and test predictions. Test scores are calculated but are not used in the threshold selection expression. Exported metrics are weighted loss, AUROC, macro-AP, and macro-F1.

This original cell writes the following under `SNOOPPI/` in the selected Drive root:

| Directory | Artifacts |
| --- | --- |
| `splits/` | `train.csv`, `validation.csv`, `test.csv` used by later model notebooks |
| `tables/` | Training history, split metrics, test predictions, and test confusion matrix CSV files |
| `figures/` | Loss, validation metrics, confusion matrix, and split comparison in PNG and PDF |
| `checkpoints/` | Selected-head checkpoint, weights, and model architecture text |
| `embeddings/` | `prot_t5_embedding_cache.pt` |

The checkpoint includes the selected epoch and threshold but does not include optimizer state, encoder weights, a pinned encoder revision, dataset revision, or a complete environment manifest. Test-prediction rows follow `test_df` order and omit pair identifiers.

**Preserved path behavior:** an existing `/content/drive/MyDrive` directory is treated as a mounted Drive without verifying the mount. Section 16 retains the historical recovery attempt for a local directory mistaken for Drive. Confirm the actual mount and intended destination before export. The cell can overwrite files with the same names, and figure-generation failure can interrupt it before checkpoint saving. No artifacts were regenerated during this cleanup.

In [ ]:
from google.colab import drive
from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score
)
import matplotlib.pyplot as plt

if Path("/content/drive/MyDrive").exists():
    MY_DRIVE = Path("/content/drive/MyDrive")
else:
    drive.mount("/content/gdrive")
    MY_DRIVE = Path("/content/gdrive/MyDrive")

ROOT = MY_DRIVE / "SNOOPPI"
CHECKPOINTS = ROOT / "checkpoints"
FIGURES = ROOT / "figures"
TABLES = ROOT / "tables"
SPLITS = ROOT / "splits"
EMBEDDINGS = ROOT / "embeddings"

for folder in [CHECKPOINTS, FIGURES, TABLES, SPLITS, EMBEDDINGS]:
    folder.mkdir(parents=True, exist_ok=True)

def collect_predictions(loader):
    model.eval()
    labels_all, probabilities_all = [], []
    total_loss, total_samples = 0.0, 0

    with torch.no_grad():
        for pair_features, labels in loader:
            pair_features = pair_features.to(DEVICE)
            labels = labels.float().view(-1).to(DEVICE)

            logits = model(pair_features).view(-1)
            loss = criterion(logits, labels)
            probabilities = torch.sigmoid(logits)

            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
            labels_all.append(labels.cpu().numpy())
            probabilities_all.append(probabilities.cpu().numpy())

    return (
        total_loss / total_samples,
        np.concatenate(labels_all).astype(int),
        np.concatenate(probabilities_all)
    )

def summarize(loss, labels, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    positive_ap = average_precision_score(labels, probabilities)
    negative_ap = average_precision_score(1 - labels, 1 - probabilities)

    return {
        "loss": loss,
        "auroc": roc_auc_score(labels, probabilities),
        "macro_ap": (positive_ap + negative_ap) / 2,
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

train_loss, train_labels, train_probabilities = collect_predictions(train_loader)
val_loss, val_labels, val_probabilities = collect_predictions(val_loader)
test_loss, test_labels, test_probabilities = collect_predictions(test_loader)

thresholds = np.linspace(0.05, 0.95, 181)
threshold = max(
    thresholds,
    key=lambda value: f1_score(
        val_labels,
        val_probabilities >= value,
        average="macro"
    )
)

results = {
    "Train": summarize(
        train_loss, train_labels, train_probabilities, threshold
    ),
    "Validation": summarize(
        val_loss, val_labels, val_probabilities, threshold
    ),
    "Test": summarize(
        test_loss, test_labels, test_probabilities, threshold
    )
}

split_metrics = (
    pd.DataFrame(results)
    .T
    .rename_axis("split")
    .reset_index()
)

test_predictions = pd.DataFrame({
    "label": test_labels,
    "interaction_score": test_probabilities,
    "prediction": (test_probabilities >= threshold).astype(int)
})

test_cm = confusion_matrix(
    test_labels,
    test_predictions["prediction"],
    labels=[0, 1]
)

history_df.to_csv(TABLES / "training_history.csv", index=False)
split_metrics.to_csv(TABLES / "split_metrics.csv", index=False)
test_predictions.to_csv(TABLES / "test_predictions.csv", index=False)

pd.DataFrame(
    test_cm,
    index=["true_negative", "true_positive"],
    columns=["predicted_negative", "predicted_positive"]
).to_csv(TABLES / "test_confusion_matrix.csv")

train_df.to_csv(SPLITS / "train.csv", index=False)
val_df.to_csv(SPLITS / "validation.csv", index=False)
test_df.to_csv(SPLITS / "test.csv", index=False)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150
})

best_epoch = int(
    history_df.loc[
        history_df["val_macro_ap"].idxmax(),
        "epoch"
    ]
)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(
    history_df["epoch"],
    history_df["train_loss"],
    color="#3366A8",
    linewidth=2.2,
    label="Training"
)
ax.plot(
    history_df["epoch"],
    history_df["val_loss"],
    color="#D55E00",
    linewidth=2.2,
    label="Validation"
)
ax.axvline(
    best_epoch,
    color="#555555",
    linestyle="--",
    linewidth=1,
    label=f"Selected epoch: {best_epoch}"
)
ax.set(
    xlabel="Epoch",
    ylabel="Weighted binary cross-entropy",
    title="Training and validation loss"
)
ax.grid(alpha=0.2)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "figure_1_loss_curves.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "figure_1_loss_curves.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(
    history_df["epoch"],
    history_df["val_auroc"],
    color="#009E73",
    linewidth=2.2,
    label="AUROC"
)
ax.plot(
    history_df["epoch"],
    history_df["val_macro_ap"],
    color="#CC79A7",
    linewidth=2.2,
    label="Macro-AP"
)
ax.axvline(
    best_epoch,
    color="#555555",
    linestyle="--",
    linewidth=1
)
ax.set(
    xlabel="Epoch",
    ylabel="Score",
    ylim=(0.5, 1.0),
    title="Validation performance"
)
ax.grid(alpha=0.2)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "figure_2_validation_metrics.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "figure_2_validation_metrics.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5.2, 4.5))
image = ax.imshow(test_cm, cmap="Blues")

for row in range(2):
    for column in range(2):
        ax.text(
            column,
            row,
            f"{test_cm[row, column]:,}",
            ha="center",
            va="center",
            color="white" if test_cm[row, column] > test_cm.max() / 2 else "black",
            fontsize=12
        )

ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["Non-interaction", "Interaction"],
    yticklabels=["Non-interaction", "Interaction"],
    xlabel="Predicted class",
    ylabel="True class",
    title=f"Test confusion matrix (threshold = {threshold:.2f})"
)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURES / "figure_3_test_confusion_matrix.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "figure_3_test_confusion_matrix.png", bbox_inches="tight")
plt.show()

metric_columns = ["auroc", "macro_ap", "macro_f1"]
metric_labels = ["AUROC", "Macro-AP", "Macro-F1"]
colors = ["#3366A8", "#E69F00", "#009E73"]
x = np.arange(len(split_metrics))
width = 0.24

fig, ax = plt.subplots(figsize=(7.0, 4.4))

for index, (column, label, color) in enumerate(
    zip(metric_columns, metric_labels, colors)
):
    ax.bar(
        x + (index - 1) * width,
        split_metrics[column],
        width,
        color=color,
        label=label
    )

ax.set(
    xticks=x,
    xticklabels=split_metrics["split"],
    ylabel="Score",
    ylim=(0.5, 1.0),
    title="Performance across data splits"
)
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
fig.savefig(FIGURES / "figure_4_split_comparison.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "figure_4_split_comparison.png", bbox_inches="tight")
plt.show()

checkpoint = {
    "selected_epoch": best_epoch,
    "model_state_dict": model.state_dict(),
    "model_class": model.__class__.__name__,
    "model_architecture": str(model),
    "decision_threshold": float(threshold),
    "loss_function": criterion.__class__.__name__,
    "split_seed": 44,
    "validation_selection_metric": "macro_ap",
    "split_metrics": split_metrics.set_index("split").to_dict("index")
}

torch.save(
    checkpoint,
    CHECKPOINTS / "best_predictor_checkpoint.pt"
)

torch.save(
    model.state_dict(),
    CHECKPOINTS / "best_predictor_weights.pt"
)

if "embedding_cache" in globals():
    torch.save(
        embedding_cache,
        EMBEDDINGS / "prot_t5_embedding_cache.pt"
    )

with open(CHECKPOINTS / "model_architecture.txt", "w") as file:
    file.write(str(model))

print(split_metrics.round(4))
print(f"\nValidation-selected threshold: {threshold:.3f}")
print(f"Selected epoch: {best_epoch}")
print(f"Saved to: {ROOT}")

## 13. Historical runtime-variable inventory

Inspect datasets and large dataframes currently stored in the notebook runtime. This debugging cell does not alter labels or train a model. It imports Hugging Face `Dataset` into the global name previously used for PyTorch `Dataset`; rerunning earlier class definitions after this cell can therefore change name resolution. It is retained in its original late position.

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, Dataset):
        print(name, "| Dataset |", f"{len(obj):,}", "|", obj.column_names)
    elif isinstance(obj, DatasetDict):
        print(name, "| DatasetDict |", {
            split: f"{len(data):,}"
            for split, data in obj.items()
        })
    elif isinstance(obj, pd.DataFrame) and len(obj) > 1000:
        print(name, "| DataFrame |", f"{len(obj):,}", "|", obj.columns.tolist())

## 14. Optional unknown-split archive and label inspection

Save the dataset currently referenced by `split_data` to `ROOT / "dataset" / "snooppi_835k"`, then print its unique SNOOPPI labels. In the supplied notebook's stored outputs, this was the **unknown-label** split, containing 835,083 pairs; it was not the supervised training set.

The code uses the residual loop variable from section 03 rather than an explicit `snooppi["unknown"]` lookup, and the label inspection occurs **after** saving. Its identity depends on runtime state and dataset iteration order. Inspect that identity before executing this optional archive; the folder name itself is not a dataset validation. This section does not use unknown pairs as negative examples.

In [ ]:
save_path = ROOT / "dataset" / "snooppi_835k"
save_path.parent.mkdir(parents=True, exist_ok=True)
split_data.save_to_disk(str(save_path), max_shard_size="2GB")
print(f"Saved {len(split_data):,} pairs to {save_path}")

In [ ]:
print(split_data.unique("SNOOPPI_final_label"))

## 15. Historical Google Drive mount diagnostic

Check whether `/content/drive` is an actual filesystem mount. This original diagnostic distinguishes a mounted Drive from an ordinary local directory with the same path. It is especially relevant because the export in section 12 checks directory existence rather than mount status.

In [ ]:
import os

print("Real Google Drive mount:", os.path.ismount("/content/drive"))

## 16. Historical manual Drive recovery

Mount Google Drive at `/content/cloud_drive`, recursively copy the existing `/content/drive/MyDrive/SNOOPPI` directory into that mounted destination, and update `ROOT`. This was a manual recovery path for artifacts saved under a local directory mistaken for Drive.

**Optional recovery only:** verify the source and destination before use. `dirs_exist_ok=True` merges into an existing destination and can overwrite matching files; this is not a routine training step. Most previously defined directory variables are not recomputed by this cell. Section 17 explicitly recalculates `TABLES` from the updated `ROOT`.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

LOCAL_ROOT = Path("/content/drive/MyDrive/SNOOPPI")

drive.mount("/content/cloud_drive")

CLOUD_ROOT = Path("/content/cloud_drive/MyDrive/SNOOPPI")
CLOUD_ROOT.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    LOCAL_ROOT,
    CLOUD_ROOT,
    dirs_exist_ok=True
)

ROOT = CLOUD_ROOT

print("Copied to real Google Drive:", ROOT)

## 17. Table existence checks and Overleaf export

Check for the history, split-metrics, and confusion-matrix CSV files under the current `ROOT`, bundle those three tables into `/content/SNOOPPI_overleaf_data.zip`, and request a browser download through Colab. The archive contains these tables only; it does not include model weights, predictions, or figure images.

In [ ]:
from pathlib import Path

TABLES = ROOT / "tables"

plot_files = [
    TABLES / "training_history.csv",
    TABLES / "split_metrics.csv",
    TABLES / "test_confusion_matrix.csv"
]

for path in plot_files:
    print(path, "| exists:", path.exists())

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/SNOOPPI_overleaf_data.zip")

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in plot_files:
        if not path.exists():
            raise FileNotFoundError(path)

        archive.write(
            path,
            arcname=path.name
        )

files.download(str(ZIP_PATH))